# Lab 5.3: Subagent Invocation & Context Passing

**What you'll build:** A system that demonstrates proper context passing to subagents -- and learn why missing or excessive context causes failures.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- dispatch a subagent with no context and watch it fail | 5 min |
| 2 | The Right Way -- pass explicit, focused context to subagents | 10 min |
| 3 | Your Turn -- design context for a new subagent task | 5 min |
| 4 | Stress Test -- compare Task (isolated) vs fork_session (shared) | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a document analysis system. The coordinator receives a document, extracts metadata, and dispatches subagents for different types of analysis. The key challenge: how much context to pass to each subagent.

We'll use a simulated research report to demonstrate context passing patterns.

In [ ]:
# Simulated research document
DOCUMENT = {
    "title": "Q4 2024 Market Analysis: AI Infrastructure Spending",
    "author": "Research Team Alpha",
    "date": "2024-12-15",
    "sections": {
        "executive_summary": "AI infrastructure spending grew 45% YoY in Q4 2024, reaching $12.8B. Cloud providers led growth at 52%, while on-premise spending grew 28%. Key drivers: LLM training demand and inference scaling.",
        "market_data": {
            "total_spending": 12.8,
            "unit": "billion_usd",
            "yoy_growth": 0.45,
            "segments": {
                "cloud": {"spending": 7.6, "growth": 0.52},
                "on_premise": {"spending": 3.8, "growth": 0.28},
                "edge": {"spending": 1.4, "growth": 0.61}
            }
        },
        "competitors": [
            {"name": "CloudCo", "market_share": 0.34, "revenue": 4.35},
            {"name": "DataScale", "market_share": 0.22, "revenue": 2.82},
            {"name": "InfraMax", "market_share": 0.18, "revenue": 2.30},
            {"name": "Others", "market_share": 0.26, "revenue": 3.33}
        ],
        "risks": "Supply chain constraints for GPU allocation may limit growth in Q1 2025. Regulatory uncertainty in EU around AI compute requirements could impact cloud segment.",
        "recommendations": "Increase allocation to edge computing (highest growth segment). Monitor GPU supply chain weekly. Prepare compliance framework for EU AI Act."
    }
}

# Analysis tools
ANALYSIS_TOOLS = [
    {
        "name": "calculate_metric",
        "description": "Calculate a derived metric from numeric inputs",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression to evaluate"},
                "label": {"type": "string", "description": "Label for the result"}
            },
            "required": ["expression", "label"]
        }
    },
    {
        "name": "flag_risk",
        "description": "Flag a risk factor with severity",
        "input_schema": {
            "type": "object",
            "properties": {
                "risk": {"type": "string"},
                "severity": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                "mitigation": {"type": "string"}
            },
            "required": ["risk", "severity", "mitigation"]
        }
    }
]

def execute_analysis_tool(name, tool_input):
    if name == "calculate_metric":
        try:
            result = eval(tool_input["expression"], {"__builtins__": {}}, {})
            return json.dumps({"label": tool_input["label"], "value": round(result, 4)})
        except Exception as e:
            return json.dumps({"error": str(e)})
    elif name == "flag_risk":
        return json.dumps({"flagged": True, "risk": tool_input["risk"], "severity": tool_input["severity"]})
    return json.dumps({"error": f"Unknown tool: {name}"})

print(f"Document: {DOCUMENT['title']}")
print(f"Sections: {list(DOCUMENT['sections'].keys())}")

---

## Phase 1: The Wrong Approach

Dispatch a subagent with NO relevant context -- just a generic task description.

In [ ]:
def run_subagent(task_prompt, tools, label="Subagent"):
    """Run a subagent with fresh context."""
    messages = [{"role": "user", "content": task_prompt}]
    
    for _ in range(5):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            final = "".join(b.text for b in response.content if hasattr(b, "text"))
            return final
        
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_analysis_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
    
    return "(hit safety cap)"


# BAD: No context passed
bad_prompt = "Analyze the market data and calculate growth metrics."

print("=== SUBAGENT WITH NO CONTEXT ===")
print(f"Prompt: {bad_prompt}\n")
bad_result = run_subagent(bad_prompt, ANALYSIS_TOOLS, "No-Context")
print(f"Result: {bad_result}")
print()
print("Problem: The subagent has no data to analyze! It either hallucinated numbers or asked for data it cannot access.")

---

## Phase 2: The Right Approach

Pass explicit, focused context to each subagent. Include only what it needs.

In [ ]:
# GOOD: Explicit context for market analysis subagent
market_data = DOCUMENT["sections"]["market_data"]

good_prompt = f"""Analyze the following market data and calculate key growth metrics.

MARKET DATA:
- Total AI infrastructure spending: ${market_data['total_spending']}B
- Year-over-year growth: {market_data['yoy_growth']*100}%
- Cloud segment: ${market_data['segments']['cloud']['spending']}B ({market_data['segments']['cloud']['growth']*100}% growth)
- On-premise segment: ${market_data['segments']['on_premise']['spending']}B ({market_data['segments']['on_premise']['growth']*100}% growth)
- Edge segment: ${market_data['segments']['edge']['spending']}B ({market_data['segments']['edge']['growth']*100}% growth)

TASK:
1. Calculate each segment's share of total spending (as a percentage)
2. Identify the fastest-growing segment
3. Calculate what total spending would be if current growth continues for another year

Use the calculate_metric tool for computations. Return a brief structured summary."""

print("=== SUBAGENT WITH EXPLICIT CONTEXT ===")
print(f"Prompt length: {len(good_prompt)} chars\n")
good_result = run_subagent(good_prompt, ANALYSIS_TOOLS, "Market Analysis")
print(f"\nResult:\n{good_result}")

In [ ]:
# Risk analysis subagent -- different context, different tools
risk_prompt = f"""Analyze the following risk factors and flag each with a severity rating.

RISK INFORMATION:
{DOCUMENT['sections']['risks']}

CONTEXT:
- This is for Q4 2024 AI infrastructure market analysis
- Total market size: $12.8B
- Cloud segment is the largest at $7.6B

TASK:
1. Identify each distinct risk factor
2. Flag each risk with severity (low/medium/high/critical) and a mitigation strategy
3. Return a structured summary of all flagged risks

Use the flag_risk tool for each risk you identify."""

print("=== RISK ANALYSIS SUBAGENT ===")
risk_result = run_subagent(risk_prompt, ANALYSIS_TOOLS, "Risk Analysis")
print(f"\nResult:\n{risk_result}")

### What Changed

| Aspect | No Context | Explicit Context |
|--------|-----------|------------------|
| Data available | None | Exactly the needed fields |
| Task clarity | Vague ("analyze market data") | Specific (calculate X, identify Y) |
| Output format | Unpredictable | Structured summary |
| Result quality | Hallucinated or empty | Accurate, grounded in real data |

---

## Phase 3: Your Turn

Design the context for a **competitor analysis** subagent. The subagent needs to analyze the competitor data and identify the market leader and its advantages.

In [ ]:
# =============================================================
# TODO: Write the competitor analysis prompt with explicit context
# =============================================================
#
# Requirements:
# - Include the competitor data from DOCUMENT["sections"]["competitors"]
# - Include relevant market context (total spending, growth rate)
# - Specify exactly what analysis to perform
# - Specify the expected output format
#
# Do NOT include: risk data, recommendations, or executive summary
# (the subagent does not need those for competitor analysis)

competitor_prompt = """
# TODO: Write your competitor analysis prompt here.
# Include: competitor data, total market size, task description, output format.
# Exclude: risks, recommendations, executive summary.
"""

# competitor_result = run_subagent(competitor_prompt, ANALYSIS_TOOLS, "Competitor Analysis")
# print(f"Result:\n{competitor_result}")

print("TODO: Complete the competitor analysis prompt and uncomment the run.")

In [ ]:
# =============================================================
# Checker: validates your context passing
# =============================================================

print("=== CONTEXT PASSING CHECKLIST ===")

checks = [
    ("Contains competitor data", any(c["name"] in competitor_prompt for c in DOCUMENT["sections"]["competitors"])),
    ("Contains total market size", "12.8" in competitor_prompt or "12800" in competitor_prompt),
    ("Has task description", len(competitor_prompt) > 100),
    ("Does NOT contain risk info", "GPU" not in competitor_prompt and "supply chain" not in competitor_prompt.lower()),
    ("Does NOT contain recommendations", "edge computing" not in competitor_prompt.lower()),
]

passed = 0
for label, result in checks:
    status = "PASS" if result else "FAIL"
    print(f"  [{status}] {label}")
    if result:
        passed += 1

print(f"\n{passed}/{len(checks)} checks passed")
if passed == len(checks):
    print("PASSED -- Your context is properly scoped: includes needed data, excludes irrelevant info.")
else:
    print("Revise your prompt to include only competitor-relevant context.")

---

## Phase 4: Task vs fork_session

Demonstrating the conceptual difference between Task (fresh context) and fork_session (inherited context).

In [ ]:
# Simulating Task vs fork_session behavior

# Scenario: After initial analysis, we want to explore two approaches
initial_analysis = "The edge segment shows 61% growth but only $1.4B in absolute spending."

# TASK approach: Fresh context, explicit data passing
print("=== TASK APPROACH (Fresh Context) ===")
task_prompt = f"""Given this finding: {initial_analysis}

Market context: Total market is $12.8B. Edge is the fastest-growing segment.

Should the firm increase edge computing investment? 
Calculate what edge spending would be next year at current growth rate.
Use the calculate_metric tool."""

task_result = run_subagent(task_prompt, ANALYSIS_TOOLS, "Task")
print(f"Result: {task_result[:200]}\n")

# FORK_SESSION approach: Would inherit FULL context
# (simulated -- in a real system, fork_session copies the entire message history)
print("=== FORK_SESSION APPROACH (Inherited Context) ===")
print("In a real system, fork_session would give the subagent:")
print("  - All prior coordinator messages")
print("  - All prior tool calls and results")
print("  - The full analysis context so far")
print("  - Same tools as the parent session")
print()
print("Use fork_session when: exploratory branches that need shared understanding")
print("Use Task when: isolated work with scoped context and tools")

### Key Takeaways

1. **Context must be explicitly passed.** Subagents (via Task) start with fresh context. They cannot "see" anything the coordinator does not include.
2. **Include only what is needed.** The market analysis subagent needs market data, not risk factors. The risk subagent needs risk info, not competitor details.
3. **Task vs fork_session are not interchangeable.** Task = isolated fresh context (for focused subagent work). fork_session = inherited full context (for exploratory branches).
4. **Treat subagent prompts like function calls.** Specify: inputs (data), task (what to do), constraints (what to skip), and output format (what to return).